# Building data pipelines 🔄

## What you will learn in this course 🧐🧐

In the previous lecture, you learned the three core skills: grouping, handling missing values, and merging. You wrote each step on its own line, with intermediate variables. That style is fine when you explore data. It becomes hard to manage when the logic grows past five or six steps. Names pile up. You forget which DataFrame is the latest one. Mistakes slip in.

A **data pipeline** is a clean, repeatable sequence of transformations that turns raw input into a final business output. In this lecture, you will learn how to build pipelines that are short, readable, and safe. You will use **step functions** (small functions that do one thing each) and **method chaining** (calling several pandas methods one after another in a single flow). 

By the end of this course, you will be able to:

- Define what a data pipeline is and explain why it matters in production.
- Write step functions that take a DataFrame and return a DataFrame.
- Use `pipe()` to plug your own functions into a chain.
- Build end-to-end pipelines using `query()`, `assign()`, `merge()`, `groupby()`, and `pivot_table()`.


In [1]:
import pandas as pd
import numpy as np

events = pd.read_csv('src/viewing_events.csv')
profiles = pd.read_csv('src/user_profiles.csv')

print(f"Loaded {len(events)} events and {len(profiles)} user profiles")

Loaded 1000 events and 50 user profiles


## Two coding styles

There are two main ways to write a sequence of pandas operations.

1. **Imperative style** (step by step). You write one line at a time. Each line stores the result in a new variable. You can print between steps to check progress. This style is great for learning and exploration.

2. **Declarative style** (method chaining). You describe the *whole* transformation as one continuous flow. Pandas runs the steps in order. No intermediate names. This style is great for production code.

<img src="https://ai-fullstack-assets.s3.eu-west-3.amazonaws.com/M02-EDA/AIFS-M02-D02-Method_chaining.png"/>

Let's see the same business task written in both styles, then compare. The idea is to compute *for each subscription type, the average completion rate on play events.* The completion rate is `watched_duration / total_duration`.

### Imperative style

In [2]:
# Step 1: keep only play events
is_play = events['event_type'] == 'play'
play_events = events[is_play]

# Step 2: compute completion rate as a new column
play_events = play_events.copy()
play_events['completion_rate'] = play_events['watched_duration'] / play_events['total_duration']

# Step 3: add user profile info
play_with_profile = play_events.merge(profiles, on='user_id', how='left')

# Step 4: average per subscription type
result_imperative = play_with_profile.groupby('subscription_type').agg(
    avg_completion=('completion_rate', 'mean'),
    play_count=('user_id', 'count')
)
result_imperative = result_imperative.round(3)

print("Imperative style result:")
print(result_imperative)

Imperative style result:
                   avg_completion  play_count
subscription_type                            
basic                       0.416          52
premium                     0.440          75
standard                    0.428          67


Four steps, four named DataFrames: `play_events`, `play_events` (overwritten), `play_with_profile`, and `result_imperative`. After 10 or 20 steps, this becomes hard to follow.

Notice the `.copy()` call on step 2. Without it, pandas may show the famous `SettingWithCopyWarning` because `play_events` is a slice of `events`. Adding `.copy()` makes the intent explicit: *this is a new DataFrame I plan to modify*.

<Note type="important">

Always call `.copy()` when you create a filtered DataFrame that you plan to modify. Without it, you risk silently changing the original DataFrame, which leads to bugs that are hard to find.

</Note>

### Method chaining style

Now the exact same logic, written as one chain.

In [3]:
result_chained = (
    events
    .query("event_type == 'play'")
    .assign(completion_rate=lambda df: df['watched_duration'] / df['total_duration'])
    .merge(profiles, on='user_id', how='left')
    .groupby('subscription_type')
    .agg(
        avg_completion=('completion_rate', 'mean'),
        play_count=('user_id', 'count')
    )
    .round(3)
)

print("Chained style result:")
print(result_chained)

Chained style result:
                   avg_completion  play_count
subscription_type                            
basic                       0.416          52
premium                     0.440          75
standard                    0.428          67


Same numbers. **Premium users complete 44% of what they play, basic users only 41.6%.** That insight is unchanged.

What is different is the structure. The whole pipeline lives inside one set of parentheses. There are no intermediate variables. You read it from top to bottom like a list of instructions:

1. Start with `events`.
2. Keep only play events with `query()`.
3. Add a new column with `assign()`.
4. Merge with profiles.
5. Group by subscription.
6. Aggregate.
7. Round.

Two new methods to learn here:

- `query("event_type == 'play'")` is a string-based filter. It is equivalent to `df[df['event_type'] == 'play']` but reads like a sentence. Use it for simple filters.
- `assign(new_col=...)` returns a copy of the DataFrame with one or more new columns added. The right side uses a `lambda` (a tiny anonymous function) that receives the current DataFrame as `df`. You write `lambda df: df['watched_duration'] / df['total_duration']` to say *take the current DataFrame and compute this column*.

<Note type="tip">

The `lambda df: ...` pattern inside `assign()` is critical. You cannot write `assign(rate=events['watched'] / events['total'])` because at that point in the chain, you may have filtered or merged. The `lambda` receives the *current* state of the DataFrame, not the original one.

</Note>

## Step functions

Method chaining works well when each operation is a built-in pandas method. But real pipelines often need **custom logic** that pandas does not offer out of the box. For example, *flag a user as a power user if they generated more than 20 events*. There is no `.flag_power_users()` method.

The solution is a **step function**. A step function follows a simple contract:

> Take a DataFrame. Do one thing. Return a DataFrame.

Here is the shape:

In [4]:
def example_step(df):
    # Make a copy so the original is not changed
    df = df.copy()
    
    # Do exactly one transformation
    # ...
    
    # Always return a DataFrame
    return df

Three rules:

- **Input is always a DataFrame.** First positional argument.
- **Output is always a DataFrame.** Even if you only modify one column.
- **Each function does one thing.** Filter, or compute, or rename. Not all three.

When all your functions follow these rules, you can connect them safely. Let's write three real step functions for our streaming data.

In [5]:
def keep_play_events(df):
    """Keep only events where the user actually played content."""
    cleaned = df.copy()
    is_play = cleaned['event_type'] == 'play'
    cleaned = cleaned[is_play]
    return cleaned


def add_completion_rate(df):
    """Add a column that measures how much of the content the user watched."""
    df = df.copy()
    watched = df['watched_duration']
    total = df['total_duration']
    df['completion_rate'] = watched / total
    return df


def add_user_info(df, user_profiles):
    """Attach user subscription type, country, and revenue to each event."""
    df = df.copy()
    enriched = df.merge(user_profiles, on='user_id', how='left')
    return enriched

Each function does one thing. Each function returns a DataFrame. Each function has a docstring (the triple-quoted string) that explains its purpose in one sentence. This is the discipline of pipeline code.

Notice that `add_user_info` takes a second argument: `user_profiles`. This is fine. The first argument must be the DataFrame, but you can pass extra inputs after it.

Now let's run them as a sequence, the imperative way.

In [6]:
# Run each step manually, one after the other
step1 = keep_play_events(events)
step2 = add_completion_rate(step1)
step3 = add_user_info(step2, profiles)

preview_cols = ['user_id', 'event_type', 'completion_rate', 'subscription_type']
print("Step-by-step output (first 5 rows):")
print(step3[preview_cols].head(5))
print(f"\nFinal row count: {len(step3)}")

Step-by-step output (first 5 rows):
    user_id event_type  completion_rate subscription_type
0  user_041       play         0.636950           premium
1  user_022       play         0.314232          standard
2  user_030       play         0.475450           premium
3  user_006       play         0.723897             basic
4  user_050       play         0.423618             basic

Final row count: 194


194 play events, each enriched with completion rate and subscription type. The pipeline works. But you still have three intermediate variables: `step1`, `step2`, `step3`. Let's get rid of them.

## Plugging custom functions into a chain with pipe()

Pandas provides a special method called `pipe()`. It takes one argument: a function. The function receives the current DataFrame and must return a DataFrame. Sound familiar? That is exactly the step function contract.

Here is the same logic, but using `pipe()` to plug each step function into a single chain.

In [7]:
result = (
    events
    .pipe(keep_play_events)
    .pipe(add_completion_rate)
    .pipe(add_user_info, profiles)
)

preview_cols = ['user_id', 'event_type', 'completion_rate', 'subscription_type']
print("Pipeline output (first 5 rows):")
print(result[preview_cols].head(5))
print(f"\nFinal row count: {len(result)}")

Pipeline output (first 5 rows):
    user_id event_type  completion_rate subscription_type
0  user_041       play         0.636950           premium
1  user_022       play         0.314232          standard
2  user_030       play         0.475450           premium
3  user_006       play         0.723897             basic
4  user_050       play         0.423618             basic

Final row count: 194


Same 194 rows. Same data. Zero intermediate variables.

How `pipe()` works:

- `.pipe(keep_play_events)` calls `keep_play_events(events)` for you. Pandas automatically passes the current DataFrame as the first argument.
- `.pipe(add_user_info, profiles)` calls `add_user_info(current_df, profiles)`. Any argument after the function name is passed along to the function as additional argument.

<Note type="important">

`pipe()` is the bridge between method chaining and your own functions. It is what lets you keep a clean chain even when your business logic is too specific to be a built-in pandas method.

</Note>

## A complete production pipeline

Now let's put everything together. The CTO wants a weekly report with two outputs:

1. A **subscription summary**: average completion rate, average rating, and play count per subscription type.
2. A **pivot table** of average watched duration by subscription type and event type.

We will write one pipeline that produces both. We define each step as a function. Then we chain them with `pipe()`.

In [8]:
def filter_real_engagement(df):
    """Keep only events that reflect real engagement (drop errors and buffers)."""
    df = df.copy()
    real_events = ['play', 'pause', 'stop']
    keep = df['event_type'].isin(real_events)
    df = df[keep]
    return df


def fill_missing_ratings(df):
    """Fill missing user ratings with the overall mean rating."""
    df = df.copy()
    mean_rating = df['user_rating'].mean()
    df['user_rating'] = df['user_rating'].fillna(mean_rating)
    return df


def add_completion_rate(df):
    """Add the completion rate column."""
    df = df.copy()
    df['completion_rate'] = df['watched_duration'] / df['total_duration']
    return df


def attach_profiles(df, user_profiles):
    """Merge user subscription and revenue info into the events."""
    df = df.copy()
    df = df.merge(user_profiles, on='user_id', how='left')
    return df

Four step functions. Each one does exactly one thing. Each one is independent. You can test any of them on its own with a small sample. You can swap one out without breaking the others. *That* is what makes a pipeline maintainable.

Now let's chain them together to produce the final report.

In [9]:
# Build the cleaned dataset using the chain
clean_data = (
    events
    .pipe(filter_real_engagement)
    .pipe(fill_missing_ratings)
    .pipe(add_completion_rate)
    .pipe(attach_profiles, profiles)
)

print(f"Cleaned dataset: {clean_data.shape[0]} rows, {clean_data.shape[1]} columns")
print("\nFirst 3 rows:")
preview_cols = ['user_id', 'event_type', 'completion_rate', 'user_rating', 'subscription_type']
print(clean_data[preview_cols].head(3))

Cleaned dataset: 599 rows, 16 columns

First 3 rows:
    user_id event_type  completion_rate  user_rating subscription_type
0  user_041       play         0.636950          3.7           premium
1  user_002      pause         0.114576          3.4          standard
2  user_001      pause         0.093377          4.8           premium


The chain reads as one sentence: *take events, filter to real engagement, fill missing ratings, add completion rate, attach profiles*. Anyone can review this in 10 seconds.

Now produce the two final reports from the cleaned data.

In [10]:
# Report 1: subscription summary
subscription_summary = clean_data.groupby('subscription_type').agg(
    avg_completion=('completion_rate', 'mean'),
    avg_rating=('user_rating', 'mean'),
    event_count=('user_id', 'count')
)
subscription_summary = subscription_summary.round(3)

print("Report 1: Subscription summary")
print(subscription_summary)

Report 1: Subscription summary
                   avg_completion  avg_rating  event_count
subscription_type                                         
basic                       0.245       2.959          166
premium                     0.260       3.091          232
standard                    0.263       3.134          201


In [11]:
# Report 2: pivot table
watch_pivot = clean_data.pivot_table(
    index='subscription_type',
    columns='event_type',
    values='watched_duration',
    aggfunc='mean'
)
watch_pivot = watch_pivot.round(1)

print("Report 2: Average watched_duration by subscription and event type")
print(watch_pivot)

Report 2: Average watched_duration by subscription and event type
event_type         pause    play   stop
subscription_type                      
basic              445.0  2334.8  391.4
premium            408.8  2439.2  486.1
standard           416.2  2709.3  402.3


**Pivot tables** reshape data so you can compare two dimensions side by side. Here, rows are subscription types, columns are event types, and values are the average watched duration in seconds.

<img src="https://ai-fullstack-assets.s3.eu-west-3.amazonaws.com/M02-EDA/AIFS-M02-D02-Pivot_table.png" />

`pivot_table()` accepts four key arguments:

- `index`: column to use as row labels (`subscription_type`).
- `columns`: column to use as column headers (`event_type`).
- `values`: numeric column to aggregate (`watched_duration`).
- `aggfunc`: how to aggregate when multiple rows share the same combo (`mean` here).

Reading the pivot:

- For `play` events, **standard** users watch the longest on average (**2,709 seconds**), about 11% more than basic users (**2,335 seconds**).
- For `pause` events, the three subscription types are very close (between **408 and 445 seconds**). Pause behavior does not depend on the subscription tier.
- For `stop` events, **premium** users watch the longest before stopping (**486 seconds**), suggesting deeper engagement.

> Two reports, eight lines of code, zero copy-paste. The pipeline is ready for production.

## Every step checks its own output 🛡️

Your pipeline is ready for production. But production data changes. Next week, the events file may arrive with a missing column, a corrupted merge, or a negative revenue. If bad data flows through silently, your report is wrong and nobody notices.

The fix is a **validation step**: a step function whose only job is to check the data, not to transform it. It follows the exact same contract as every other step. DataFrame in, DataFrame out. If the data looks right, it returns it unchanged. If not, it stops the pipeline with a clear error.

The tool for this is `assert`, a Python statement that raises an `AssertionError` (an exception that stops the program and displays your message) whenever its condition is false. One assert per expectation. Let's write a validation step for our cleaned dataset.

In [ ]:
def check_clean_data(df):
    """Verify the cleaned dataset meets our expectations, or stop the pipeline."""
    # Expectation 1: the columns the reports depend on are present
    required_cols = ['user_id', 'completion_rate', 'subscription_type', 'monthly_revenue']
    missing = [col for col in required_cols if col not in df.columns]
    assert not missing, f"Missing expected columns: {missing}"

    # Expectation 2: revenue is an amount of money, it can never be negative
    negative_revenue = (df['monthly_revenue'] < 0).sum()
    assert negative_revenue == 0, f"Found {negative_revenue} rows with a negative monthly_revenue"

    # Expectation 3: no duplicated rows (a duplicate usually means a bad merge)
    duplicated_rows = df.duplicated().sum()
    assert duplicated_rows == 0, f"Found {duplicated_rows} duplicated rows (same event counted twice)"

    # All good: pass the DataFrame along, untouched
    return df

In [ ]:
# Same chain as before, with the validation step plugged at the end
clean_data = (
    events
    .pipe(filter_real_engagement)
    .pipe(fill_missing_ratings)
    .pipe(add_completion_rate)
    .pipe(attach_profiles, profiles)
    .pipe(check_clean_data)
)

print(f"All checks passed: {clean_data.shape[0]} validated rows")

In [ ]:
# What happens when bad data shows up? Let's corrupt one profile on purpose.
broken_profiles = profiles.copy()
broken_profiles.loc[0, 'monthly_revenue'] = -4.99

try:
    (
        events
        .pipe(filter_real_engagement)
        .pipe(fill_missing_ratings)
        .pipe(add_completion_rate)
        .pipe(attach_profiles, broken_profiles)
        .pipe(check_clean_data)
    )
except AssertionError as error:
    print(f"Pipeline stopped: {error}")

One corrupted profile poisoned 14 event rows after the merge, and the pipeline caught it. It refuses to produce a report from bad data. A loud failure now is far better than a silent error in the CTO's numbers next Monday.

<Note type="tip">

In production, these checks run automatically on every single execution of the pipeline, not just the day you wrote them. Mature pipelines do not even crash on a few bad rows: the invalid records are set aside in a quarantine zone for later inspection, a pattern called the Dead Letter Queue that you will see in M03. Frameworks like Pandera turn these handwritten asserts into reusable, industrial-grade validation schemas.

</Note>

## Resources 📚📚

- [DataFrame.pipe](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.pipe.html)
- [DataFrame.assign](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.assign.html)
- [DataFrame.pivot_table](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.pivot_table.html)
- [What is pandas.pipe() and Why Should You Use It?](https://medium.com/@amit25173/what-is-pandas-pipe-and-why-should-you-use-it-ec62281f6a15)